In [1]:
import draw.paper2.simulate_orbit_pair as simulate_orbit_pair
import numpy as np
from pathlib import Path


In [2]:
FIGURE_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
import importlib
importlib.reload(simulate_orbit_pair)

<module 'draw.paper2.simulate_orbit_pair' from 'C:\\usrspace\\mywork\\generic\\draw\\paper2\\simulate_orbit_pair.py'>

In [45]:
alpha_1 = 5
alpha_2 = 10
dx = 7
d_theta = 10

# 路径配置



# 自动创建目录防止报错

output_csv = FIGURE_DIR / f'orbit_pair_a{alpha_1:.0f}_a{alpha_2:.0f}.csv'

time, hops, sats1, sats2 = simulate_orbit_pair.simulate_orbit_pair(
    alpha_1_deg=alpha_1,
    alpha_2_deg=alpha_2,
    Delta_x=dx,
    theta_diff_deg=d_theta,
    output_file=output_csv
)

print(f"\n仿真完成！数据点数: {len(time)}")

仿真配置:
  α₁ = 5.00°, α₂ = 10.00°
  Δx = 7, Δθ = 10.00°
  拓扑周期 = 176.67 s, 步数 = 177
调试信息 - EC2 窗口: Raw[5.00, 25.00] -> Norm[5.00, 25.00]
统计结果: 均值=11.1525, 范围=[10.5000, 11.5000]
数据已保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\orbit_pair_a5_a10.csv

仿真完成！数据点数: 177


In [ ]:
# 定义扫描范围
alpha_max = 10.63  # 最大覆盖角
# alphas = np.linspace(0, alpha_max, 11)  # 0°, 1°, ..., 10.63°


dx=8
d_theta=20


alphas = [1,2,3,4,5,6,7,8,9,10]
results = {}


for a1 in alphas:
    for a2 in alphas:
        if a1 == 0 or a2 == 0:
            continue  # 跳过无卫星情况

        output_csv = FIGURE_DIR / f'orbit_pair_a{a1:.0f}_a{a2:.0f}.csv'



        time, hops, sats1, sats2 = simulate_orbit_pair.simulate_orbit_pair(
                alpha_1_deg=a1,
                alpha_2_deg=a2,
                Delta_x=dx,
                theta_diff_deg=d_theta,
                output_file=output_csv
            )


        print(f"仿真 α₁={a1:.1f}°, α₂={a2:.1f}°...")





In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
# 假设你已经导入了仿真模块
# sheng cheng duo zu

# --- 1. 配置路径 ---
# 请修改为你实际的保存路径
FIGURE_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

Time_DIR = FIGURE_DIR / 'time'
Time_DIR.mkdir(parents=True, exist_ok=True)
# --- 2. 参数定义 ---
alpha_max = 10.63
# alphas = np.linspace(1, 10, 10) # 如果需要浮点数
alphas = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

dx_all = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
d_theta_all = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]

print(f">> 开始参数扫描...")
print(f"   Dx范围: {dx_all}")
print(f"   Theta范围: {d_theta_all}")
print(f"   结果将保存至: {FIGURE_DIR}")

# --- 3. 循环扫描 ---
for dx in dx_all:
    for d_theta in d_theta_all:

        # 为每一组 (dx, d_theta) 创建一个结果列表
        current_results = []

        print(f"\n正在处理: Dx={dx}, D_Theta={d_theta}° ...")

        for a1 in alphas:
            for a2 in alphas:
                # 排除 0 度 (如果有的话)
                if a1 == 0 or a2 == 0:
                    continue

                try:
                    # 执行仿真
                    output_csv = Time_DIR / f'OrbitPair_dX-{dx}_dTheta-{d_theta:.2f}_Alpha-{a1:.0f}-{a2:.0f}.csv'

                    time, hops, sats1, sats2 = simulate_orbit_pair.simulate_orbit_pair(
                        alpha_1_deg=a1,
                        alpha_2_deg=a2,
                        Delta_x=dx,
                        theta_diff_deg=d_theta,
                         output_file=output_csv
                    )

                    # --- 计算跳数波动 ---
                    # 使用 numpy 处理，以防 hops 是 list
                    hops_arr = np.array(hops)

                    # 过滤掉 NaN (断连时刻不参与波动的计算，或者根据需求视断连为无穷大)
                    valid_hops = hops_arr[~np.isnan(hops_arr)]


                                        # 在你的循环中添加
                    if len(valid_hops) > 0:
                        h_mean = np.mean(valid_hops)
                        h_std = np.std(valid_hops)
                        h_cv = (h_std / h_mean) * 100 if h_mean > 0 else 0  # 相对误差 %
                        h_range = np.max(valid_hops) - np.min(valid_hops)
                                                # --- 覆盖率（经验频率） ---
                        # 以周期均值 h_mean 作为“典型水平” mu
                        mu = h_mean

                        eps_025 = 0.25
                        eps_05 = 0.5

                        cov_025 = np.mean(np.abs(valid_hops - mu) <= eps_025)  # in [0,1]
                        cov_05  = np.mean(np.abs(valid_hops - mu) <= eps_05)   # in [0,1]

                        # （可选）也可以输出百分比
                        cov_025_percent = cov_025 * 100
                        cov_05_percent  = cov_05 * 100
                    else:
                        h_mean = np.nan
                        h_std = np.nan
                        h_cv = np.nan
                        h_range = np.nan
                        cov_025 = np.nan
                        cov_05 = np.nan
                        cov_025_percent = np.nan
                        cov_05_percent = np.nan
                    current_results.append({
                        'alpha1': a1,
                        'alpha2': a2,
                        'range_hops': h_range,
                        'mean_hops': h_mean,
                        'std_hops': h_std,
                        'cv_percent': h_cv,          # 相对误差(%)
                        'cov_025': cov_025,          # 覆盖率（0~1）：|H-mu|<=0.25
                        'cov_05': cov_05,            # 覆盖率（0~1）：|H-mu|<=0.5
                        'cov_025_percent': cov_025_percent,  # 覆盖率（%）
                        'cov_05_percent': cov_05_percent,    # 覆盖率（%）
                    })




                except Exception as e:
                    print(f"[错误] a1={a1}, a2={a2} 仿真失败: {e}")

        # --- 4. 保存当前 (dx, d_theta) 组合的所有结果 ---
        if current_results:
            df = pd.DataFrame(current_results)

            # 文件名使用当前循环变量 dx 和 d_theta
            csv_name = f'scan_result_dx{dx}_dtheta{d_theta}.csv'
            save_path = FIGURE_DIR / csv_name

            # index=False 不保存行号
            df.to_csv(save_path, index=False)
            print(f"   -> 已保存: {csv_name} (包含 {len(df)} 条数据)")
        else:
            print("   -> 无有效数据，跳过保存。")

print("\n>> 所有扫描任务完成！")

>> 开始参数扫描...
   Dx范围: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   Theta范围: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
   结果将保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan

正在处理: Dx=3, D_Theta=0° ...
仿真配置:
  α₁ = 1.00°, α₂ = 1.00°
  Δx = 3, Δθ = 0.00°
  拓扑周期 = 176.67 s, 步数 = 177
调试信息 - EC2 窗口: Raw[0.00, 2.00] -> Norm[0.00, 2.00]
警告: 无有效连接路径
数据已保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan\time\OrbitPair_dX-3_dTheta-0.00_Alpha-1-1.csv
仿真配置:
  α₁ = 1.00°, α₂ = 2.00°
  Δx = 3, Δθ = 0.00°
  拓扑周期 = 176.67 s, 步数 = 177
调试信息 - EC2 窗口: Raw[-1.00, 3.00] -> Norm[359.00, 3.00]
警告: 无有效连接路径
数据已保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan\time\OrbitPair_dX-3_dTheta-0.00_Alpha-1-2.csv
仿真配置:
  α₁ = 1.00°, α₂ = 3.00°
  Δx = 3, Δθ = 0.00°
  拓扑周期 = 176.67 s, 步数 = 177
调试信息 - EC2 窗口: Raw[-2.00, 4.00] -> Norm[358.00, 4.00]
统计结果: 均值=4.0000, 范围=[4.0000, 4.0000]
数据已保存至: C:\usrspace\mywork\data_paper2\visibile_data\test

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # 专门用于画热力图的库
from pathlib import Path
import re
# --- 1. 配置路径 ---
# 这里设置为你存放 "scan_result_xxx.csv" 的那个文件夹
DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan")
PLOT_DIR = DATA_DIR / "heatmaps"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


alphas = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

dx_all = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
d_theta_all = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]

print(f">> 开始参数扫描...")
print(f"   Dx范围: {dx_all}")
print(f"   Theta范围: {d_theta_all}")
print(f"   结果将保存至: {FIGURE_DIR}")

# --- 3. 循环扫描 ---
for dx in dx_all:
    for d_theta in d_theta_all:

        csv_name = f'scan_result_dx{dx}_dtheta{d_theta}.csv'
        file_path = FIGURE_DIR / csv_name


        # 读取 CSV
        df = pd.read_csv(file_path)

        # 检查必要的列
        required_cols = ['alpha1', 'alpha2', 'range_hops']
        if not all(col in df.columns for col in required_cols):
            print(f"[跳过] {file_path.name} 列名不完整")
            continue

        # --- 3. 数据处理：将长表转换为矩阵 (Pivot) ---
        # index=行(Y轴), columns=列(X轴), values=颜色值
        # 这里我们画 delta_hops (跳数波动)
        pivot_table = df.pivot(index='alpha2', columns='alpha1', values='range_hops')

        # 按照索引排序，确保坐标轴是 1,2,3... 顺序排列
        pivot_table.sort_index(axis=0, ascending=False, inplace=True) # Y轴从上到下：大->小 (符合通常坐标习惯)
        pivot_table.sort_index(axis=1, ascending=True, inplace=True)  # X轴从左到右：小->大

        # --- 4. 绘图 ---
        plt.figure(figsize=(10, 8))

        # 绘制热力图
        # annot=True: 在格子里显示数值
        # fmt=".1f": 数值保留1位小数
        # cmap="RdYlBu_r": 颜色方案 (红-黄-蓝 反转)，红色表示波动大，蓝色表示稳定
        # vmin=0: 最小值固定为0
        sns.heatmap(pivot_table, annot=True, fmt=".1f",
                    cmap="RdYlBu_r", linewidths=.5, vmin=0)

        # 尝试从文件名提取 dx 和 theta 信息用于标题
        # 假设文件名格式: scan_result_dx15_dtheta0.csv
        fname = file_path.stem
        match = re.search(r'dx(\d+)_dtheta(\d+)', fname)
        if match:
            title_str = f'Hop Fluctuation (Delta)\nDx={match.group(1)}, D_Theta={match.group(2)}°'
        else:
            title_str = f'Hop Fluctuation: {fname}'

        plt.title(title_str, fontsize=14)
        plt.xlabel('Alpha 1 (deg)', fontsize=12)
        plt.ylabel('Alpha 2 (deg)', fontsize=12)

        # --- 5. 保存 ---
        save_name = f'heatmap_{file_path.stem}.png'
        plt.savefig(PLOT_DIR / save_name, dpi=150, bbox_inches='tight')
        plt.close() # 释放内存






>> 开始参数扫描...
   Dx范围: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   Theta范围: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
   结果将保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan


In [11]:
## single file

import pandas as pd
import numpy as np
from pathlib import Path
# 假设你已经导入了仿真模块


# --- 1. 配置路径 ---
# 请修改为你实际的保存路径
FIGURE_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\test")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. 参数定义 ---
alpha_max = 10.63
# alphas = np.linspace(1, 10, 10) # 如果需要浮点数
alphas1 = [  5 ]
alphas2 = [10]
dx_all = [  7 ]
d_theta_all = [ 10 ]

print(f">> 开始参数扫描...")
print(f"   Dx范围: {dx_all}")
print(f"   Theta范围: {d_theta_all}")
print(f"   结果将保存至: {FIGURE_DIR}")

# --- 3. 循环扫描 ---
for dx in dx_all:
    for d_theta in d_theta_all:

        # 为每一组 (dx, d_theta) 创建一个结果列表
        current_results = []

        output_csv = FIGURE_DIR / f'orbit_pair_a{alpha_1:.0f}_a{alpha_2:.0f}.csv'

        print(f"\n正在处理: Dx={dx}, D_Theta={d_theta}° ...")

        for a1 in alphas1:
            for a2 in alphas2:
                # 排除 0 度 (如果有的话)
                if a1 == 0 or a2 == 0:
                    continue

                try:
                    # 执行仿真
                    time, hops, sats1, sats2 = simulate_orbit_pair.simulate_orbit_pair(
                        alpha_1_deg=a1,
                        alpha_2_deg=a2,
                        Delta_x=dx,
                        theta_diff_deg=d_theta,
                        output_file=output_csv
                    )

                    # --- 计算跳数波动 ---
                    # 使用 numpy 处理，以防 hops 是 list
                    hops_arr = np.array(hops)

                    # 过滤掉 NaN (断连时刻不参与波动的计算，或者根据需求视断连为无穷大)
                    valid_hops = hops_arr[~np.isnan(hops_arr)]


                                        # 在你的循环中添加
                    if len(valid_hops) > 0:
                        h_mean = np.mean(valid_hops)
                        h_std = np.std(valid_hops)
                        h_cv = (h_std / h_mean) * 100 if h_mean > 0 else 0  # 相对误差 %
                        h_range = np.max(valid_hops) - np.min(valid_hops)
                                                # --- 覆盖率（经验频率） ---
                        # 以周期均值 h_mean 作为“典型水平” mu
                        mu = h_mean

                        eps_025 = 0.25
                        eps_05 = 0.5

                        cov_025 = np.mean(np.abs(valid_hops - mu) <= eps_025)  # in [0,1]
                        cov_05  = np.mean(np.abs(valid_hops - mu) <= eps_05)   # in [0,1]

                        # （可选）也可以输出百分比
                        cov_025_percent = cov_025 * 100
                        cov_05_percent  = cov_05 * 100
                    else:
                        h_mean = np.nan
                        h_std = np.nan
                        h_cv = np.nan
                        h_range = np.nan
                        cov_025 = np.nan
                        cov_05 = np.nan
                        cov_025_percent = np.nan
                        cov_05_percent = np.nan
                    current_results.append({
                        'alpha1': a1,
                        'alpha2': a2,
                        'range_hops': h_range,
                        'mean_hops': h_mean,
                        'std_hops': h_std,
                        'cv_percent': h_cv,          # 相对误差(%)
                        'cov_025': cov_025,          # 覆盖率（0~1）：|H-mu|<=0.25
                        'cov_05': cov_05,            # 覆盖率（0~1）：|H-mu|<=0.5
                        'cov_025_percent': cov_025_percent,  # 覆盖率（%）
                        'cov_05_percent': cov_05_percent,    # 覆盖率（%）
                    })




                except Exception as e:
                    print(f"[错误] a1={a1}, a2={a2} 仿真失败: {e}")

        # --- 4. 保存当前 (dx, d_theta) 组合的所有结果 ---
        if current_results:
            df = pd.DataFrame(current_results)

            # 文件名使用当前循环变量 dx 和 d_theta
            csv_name = f'scan_result_dx{dx}_dtheta{d_theta}.csv'
            save_path = FIGURE_DIR / csv_name

            # index=False 不保存行号
            df.to_csv(save_path, index=False)
            print(f"   -> 已保存: {csv_name} (包含 {len(df)} 条数据)")
        else:
            print("   -> 无有效数据，跳过保存。")
# --- 2. 循环处理每个文件 ---
for a1 in alphas:
    for a2 in alphas:
        csv_name = f'orbit_pair_a{a1:.0f}_a{a2:.0f}.csv'
        file_path = FIGURE_DIR / csv_name

        if not file_path.exists():
            continue

        try:
            # 读取数据
            df = pd.read_csv(file_path, comment='#', sep=',', skipinitialspace=True)

            # 检查列名是否包含所需的三个数据列
            required_cols = ['Time_s', 'Hop_Count', 'EC1_Sats', 'EC2_Sats']
            if not all(col in df.columns for col in required_cols):
                print(f"[跳过] {csv_name} 列名不完整。检测到的列名: {df.columns.tolist()}")
                continue

            # --- 3. 绘图 (修改为3个子图) ---
            # figsize=(宽, 高)，高度设大一点以容纳3个图
            # sharex=True 表示共享时间轴，只在最下面显示时间
            fig, ax = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

            # --- 子图 1: 轨道 1 卫星数 (EC1_Sats) ---
            ax[0].plot(df['Time_s'], df['EC1_Sats'], 'r.-', linewidth=1, markersize=3, label='Orbit 1 Sats')
            ax[0].set_ylabel('Sat Count (Orbit 1)')
            ax[0].set_title(f'Analysis for a1={a1:.0f}°, a2={a2:.0f}°', fontsize=14) # 总标题放在第一个图上面
            ax[0].grid(True, linestyle='--', alpha=0.6)
            ax[0].legend(loc='upper right')

            # --- 子图 2: 轨道 2 卫星数 (EC2_Sats) ---
            ax[1].plot(df['Time_s'], df['EC2_Sats'], 'g.-', linewidth=1, markersize=3, label='Orbit 2 Sats')
            ax[1].set_ylabel('Sat Count (Orbit 2)')
            ax[1].grid(True, linestyle='--', alpha=0.6)
            ax[1].legend(loc='upper right')

            # --- 子图 3: 跳数 (Hop_Count) - 原有功能 ---
            # 处理 Hop_Count 中的 nan (断连部分 matplotlib 会自动断开线条，这是符合预期的)
            ax[2].plot(df['Time_s'], df['Hop_Count'], 'b.-', linewidth=1, markersize=3, label='Hop Count')
            ax[2].set_ylabel('Hop Count')
            ax[2].set_xlabel('Time (s)') # 只有最下面的图显示 Time 标签
            ax[2].grid(True, linestyle='--', alpha=0.6)
            ax[2].legend(loc='upper right')

            # 自动调整布局，防止重叠
            plt.tight_layout()

            # --- 4. 保存 ---
            save_name = f'plot_a{a1:.0f}_a{a2:.0f}.png'
            plt.savefig(PLOT_DIR / save_name, dpi=150)
            plt.close() # 必须关闭，否则内存溢出

            count += 1
            if count % 10 == 0:
                print(f"进度: {count}/{total} ...")

        except Exception as e:
            print(f"[出错] 处理 {csv_name} 失败: {e}")

print("\n>> 所有扫描任务完成！")

>> 开始参数扫描...
   Dx范围: [7]
   Theta范围: [10]
   结果将保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\test


NameError: name 'alpha_1' is not defined

In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # 专门用于画热力图的库
from pathlib import Path
import re

# --- 1. 配置路径 ---
# 这里设置为你存放 "scan_result_xxx.csv" 的那个文件夹

# --- 1. 配置路径 ---
# 请修改为你实际的保存路径
FIGURE_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\test")



alphas1 = [  5 ]
alphas2 = [10]




# --- 2. 循环处理每个文件 ---
for a1 in alphas1:
    for a2 in alphas2:
        csv_name = f'orbit_pair_a{a1:.0f}_a{a2:.0f}.csv'
        file_path = FIGURE_DIR / csv_name


        df = pd.read_csv(file_path)

        # 检查必要的列
        required_cols = ['alpha1', 'alpha2', 'cov_05']
        if not all(col in df.columns for col in required_cols):
            print(f"[跳过] {file_path.name} 列名不完整")
            continue

        # --- 3. 数据处理：将长表转换为矩阵 (Pivot) ---
        # index=行(Y轴), columns=列(X轴), values=颜色值
        # 这里我们画 delta_hops (跳数波动)
        pivot_table = df.pivot(index='alpha2', columns='alpha1', values='cov_05')

        # 按照索引排序，确保坐标轴是 1,2,3... 顺序排列
        pivot_table.sort_index(axis=0, ascending=False, inplace=True) # Y轴从上到下：大->小 (符合通常坐标习惯)
        pivot_table.sort_index(axis=1, ascending=True, inplace=True)  # X轴从左到右：小->大

        # --- 4. 绘图 ---
        plt.figure(figsize=(10, 8))

        # 绘制热力图
        # annot=True: 在格子里显示数值
        # fmt=".1f": 数值保留1位小数
        # cmap="RdYlBu_r": 颜色方案 (红-黄-蓝 反转)，红色表示波动大，蓝色表示稳定
        # vmin=0: 最小值固定为0
        sns.heatmap(pivot_table, annot=True, fmt=".1f",
                    cmap="RdYlBu_r", linewidths=.5, vmin=0)

        # 尝试从文件名提取 dx 和 theta 信息用于标题
        # 假设文件名格式: scan_result_dx15_dtheta0.csv
        fname = file_path.stem
        match = re.search(r'dx(\d+)_dtheta(\d+)', fname)
        if match:
            title_str = f'Hop Fluctuation (Delta)\nDx={match.group(1)}, D_Theta={match.group(2)}°'
        else:
            title_str = f'Hop Fluctuation: {fname}'

        plt.title(title_str, fontsize=14)
        plt.xlabel('Alpha 1 (deg)', fontsize=12)
        plt.ylabel('Alpha 2 (deg)', fontsize=12)

        # --- 5. 保存 ---
        save_name = f'heatmap_{file_path.stem}.png'
        plt.savefig(PLOT_DIR / save_name, dpi=150, bbox_inches='tight')
        plt.close() # 释放内存

        count += 1
        print(f"[{count}/{total}] 已生成: {save_name}")


print(f"\n>> 全部完成！共生成 {count} 张热力图。")

[跳过] orbit_pair_a5_a10.csv 列名不完整

>> 全部完成！共生成 2 张热力图。


In [10]:

## huizhi tiaoshu tu
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
# 这里设置为你存放 "scan_result_xxx.csv" 的那个文件夹
DATA_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan")
PLOT_DIR = DATA_DIR / "heatmaps"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


alphas = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

dx_all = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
d_theta_all = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]

print(f">> 开始参数扫描...")
print(f"   Dx范围: {dx_all}")
print(f"   Theta范围: {d_theta_all}")
print(f"   结果将保存至: {FIGURE_DIR}")

# --- 3. 循环扫描 ---
for dx in dx_all:
    for d_theta in d_theta_all:

        csv_name = f'scan_result_dx{dx}_dtheta{d_theta}.csv'
        file_path = FIGURE_DIR / csv_name





        file_path = FIGURE_DIR / csv_name

        if not file_path.exists():
            continue

        try:
            # 读取数据
            df = pd.read_csv(file_path, comment='#', sep=',', skipinitialspace=True)

            # 检查列名是否包含所需的三个数据列
            required_cols = ['Time_s', 'Hop_Count', 'EC1_Sats', 'EC2_Sats']
            if not all(col in df.columns for col in required_cols):
                print(f"[跳过] {csv_name} 列名不完整。检测到的列名: {df.columns.tolist()}")
                continue

            # --- 3. 绘图 (修改为3个子图) ---
            # figsize=(宽, 高)，高度设大一点以容纳3个图
            # sharex=True 表示共享时间轴，只在最下面显示时间
            fig, ax = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

            # --- 子图 1: 轨道 1 卫星数 (EC1_Sats) ---
            ax[0].plot(df['Time_s'], df['EC1_Sats'], 'r.-', linewidth=1, markersize=3, label='Orbit 1 Sats')
            ax[0].set_ylabel('Sat Count (Orbit 1)')
            ax[0].set_title(f'Analysis for a1={a1:.0f}°, a2={a2:.0f}°', fontsize=14) # 总标题放在第一个图上面
            ax[0].grid(True, linestyle='--', alpha=0.6)
            ax[0].legend(loc='upper right')

            # --- 子图 2: 轨道 2 卫星数 (EC2_Sats) ---
            ax[1].plot(df['Time_s'], df['EC2_Sats'], 'g.-', linewidth=1, markersize=3, label='Orbit 2 Sats')
            ax[1].set_ylabel('Sat Count (Orbit 2)')
            ax[1].grid(True, linestyle='--', alpha=0.6)
            ax[1].legend(loc='upper right')

            # --- 子图 3: 跳数 (Hop_Count) - 原有功能 ---
            # 处理 Hop_Count 中的 nan (断连部分 matplotlib 会自动断开线条，这是符合预期的)
            ax[2].plot(df['Time_s'], df['Hop_Count'], 'b.-', linewidth=1, markersize=3, label='Hop Count')
            ax[2].set_ylabel('Hop Count')
            ax[2].set_xlabel('Time (s)') # 只有最下面的图显示 Time 标签
            ax[2].grid(True, linestyle='--', alpha=0.6)
            ax[2].legend(loc='upper right')

            # 自动调整布局，防止重叠
            plt.tight_layout()

            # --- 4. 保存 ---
            save_name = f'plot_a{a1:.0f}_a{a2:.0f}.png'
            plt.savefig(PLOT_DIR / save_name, dpi=150)
            plt.close() # 必须关闭，否则内存溢出

            count += 1


        except Exception as e:
            print(f"[出错] 处理 {csv_name} 失败: {e}")

print(f"\n>> 全部完成！共生成 {count} 张图片。")

>> 开始参数扫描...
   Dx范围: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   Theta范围: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
   结果将保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit

>> 全部完成！共生成 0 张图片。


In [ ]:

## huizhi tiaoshu tu
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# --- 1. 配置路径 ---
# 请确保路径存在，或者根据你的实际情况修改
FIGURE_DIR = Path(r"C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan")
PLOT_DIR = FIGURE_DIR / "hopsplots"
PLOT_DIR.mkdir(parents=True, exist_ok=True) # parents=True 防止父目录不存在报错

Time_DIR = FIGURE_DIR / "time"

# --- 2. 参数定义 ---
alpha_max = 10.63
# alphas = np.linspace(1, 10, 10) # 如果需要浮点数
alphas = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

dx_all = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
d_theta_all = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]

print(f">> 开始参数扫描...")
print(f"   Dx范围: {dx_all}")
print(f"   Theta范围: {d_theta_all}")
print(f"   结果将保存至: {FIGURE_DIR}")

# --- 3. 循环扫描 ---
for dx in dx_all:
    for d_theta in d_theta_all:

        # 为每一组 (dx, d_theta) 创建一个结果列表
        current_results = []

        print(f"\n正在处理: Dx={dx}, D_Theta={d_theta}° ...")

        for a1 in alphas:
         for a2 in alphas:


            file_path = Time_DIR / f'OrbitPair_dX-{dx}_dTheta-{d_theta:.2f}_Alpha-{a1:.0f}-{a2:.0f}.csv'
            # csv_name = f'orbit_pair_a{a1:.0f}_a{a2:.0f}.csv'
            # file_path = FIGURE_DIR / csv_name

            if not file_path.exists():
                continue


            # 读取数据
            df = pd.read_csv(file_path, comment='#', sep=',', skipinitialspace=True)

            # 检查列名是否包含所需的三个数据列
            required_cols = ['Time_s', 'Hop_Count', 'EC1_Sats', 'EC2_Sats']
            if not all(col in df.columns for col in required_cols):
                print(f"[跳过] {csv_name} 列名不完整。检测到的列名: {df.columns.tolist()}")
                continue

            # --- 3. 绘图 (修改为3个子图) ---
            # figsize=(宽, 高)，高度设大一点以容纳3个图
            # sharex=True 表示共享时间轴，只在最下面显示时间
            fig, ax = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

            # --- 子图 1: 轨道 1 卫星数 (EC1_Sats) ---
            ax[0].plot(df['Time_s'], df['EC1_Sats'], 'r.-', linewidth=1, markersize=3, label='Orbit 1 Sats')
            ax[0].set_ylabel('Sat Count (Orbit 1)')
            ax[0].set_title(f'Analysis for a1={a1:.0f}°, a2={a2:.0f}°', fontsize=14) # 总标题放在第一个图上面
            ax[0].grid(True, linestyle='--', alpha=0.6)
            ax[0].legend(loc='upper right')

            # --- 子图 2: 轨道 2 卫星数 (EC2_Sats) ---
            ax[1].plot(df['Time_s'], df['EC2_Sats'], 'g.-', linewidth=1, markersize=3, label='Orbit 2 Sats')
            ax[1].set_ylabel('Sat Count (Orbit 2)')
            ax[1].grid(True, linestyle='--', alpha=0.6)
            ax[1].legend(loc='upper right')

            # --- 子图 3: 跳数 (Hop_Count) - 原有功能 ---
            # 处理 Hop_Count 中的 nan (断连部分 matplotlib 会自动断开线条，这是符合预期的)
            ax[2].plot(df['Time_s'], df['Hop_Count'], 'b.-', linewidth=1, markersize=3, label='Hop Count')
            ax[2].set_ylabel('Hop Count')
            ax[2].set_xlabel('Time (s)') # 只有最下面的图显示 Time 标签
            ax[2].grid(True, linestyle='--', alpha=0.6)
            ax[2].legend(loc='upper right')

            # 自动调整布局，防止重叠
            plt.tight_layout()




            # --- 4. 保存 ---
            save_name = f'plot_dX-{dx}_dTheta-{d_theta:.2f}_Alpha-{a1:.0f}-{a2:.0f}.png'
            plt.savefig(PLOT_DIR / save_name, dpi=150)
            plt.close() # 必须关闭，否则内存溢出




>> 开始参数扫描...
   Dx范围: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
   Theta范围: [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
   结果将保存至: C:\usrspace\mywork\data_paper2\visibile_data\test\orbit\delta_scan

正在处理: Dx=3, D_Theta=0° ...
